# 🥉 Camada Bronze — Raw Ingest

A camada Bronze é o ponto de entrada do pipeline. Ela recebe os dados **exatamente como chegam da fonte**, sem nenhuma transformação, preservando o histórico bruto para auditoria e reprocessamento.

**Fonte:** Sistema SCADA industrial (simulado via VALUES)

**Responsabilidades desta camada:**
- Ingerir dados brutos de equipamentos industriais
- Adicionar metadado de ingestão (`ingested_at`)
- Preservar anomalias e valores inválidos para tratamento posterior
- Armazenar em formato Delta Lake para garantir ACID e time travel

> ⚠️ Os dados contêm anomalias **intencionais**: `equipment_id` nulo, `production_qty` nulo e `downtime_minutes` negativo. Esses casos serão tratados na camada Silver.

## 1. Criando o schema Bronze

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;

## 2. Ingestão dos dados brutos

Simulação de leitura de um sistema SCADA com 15 registros de eventos de produção industrial.

| Campo | Tipo | Descrição |
|---|---|---|
| `equipment_id` | STRING | ID do equipamento (pode ser nulo) |
| `line_id` | STRING | Linha de produção |
| `shift` | STRING | Turno (A, B ou C) |
| `event_timestamp` | STRING | Data/hora do evento |
| `production_qty` | INT | Quantidade produzida (pode ser nulo) |
| `downtime_minutes` | INT | Tempo de parada em minutos (pode ser negativo) |
| `defect_qty` | INT | Quantidade de defeitos |
| `source_system` | STRING | Sistema de origem |
| `ingested_at` | TIMESTAMP | Timestamp de ingestão |

In [0]:
%sql
CREATE OR REPLACE TABLE bronze.production_events AS
SELECT
  equipment_id,
  line_id,
  shift,
  event_timestamp,
  production_qty,
  downtime_minutes,
  defect_qty,
  source_system,
  current_timestamp() AS ingested_at
FROM (
  VALUES
    -- Registros válidos
    ('EQ-001', 'LINE-A', 'A', '2026-01-01 06:00:00', 210,  5,  3, 'SCADA'),
    ('EQ-002', 'LINE-A', 'B', '2026-01-01 14:00:00', 195, 12,  7, 'SCADA'),
    ('EQ-003', 'LINE-B', 'A', '2026-01-01 06:00:00', 230,  0,  2, 'SCADA'),
    ('EQ-001', 'LINE-A', 'C', '2026-01-01 22:00:00', 180, 20,  9, 'SCADA'),
    ('EQ-004', 'LINE-B', 'B', '2026-01-02 14:00:00', 205,  8,  4, 'SCADA'),
    ('EQ-005', 'LINE-C', 'A', '2026-01-02 06:00:00', 220,  3,  1, 'SCADA'),
    ('EQ-002', 'LINE-C', 'C', '2026-01-02 22:00:00', 190, 15,  6, 'SCADA'),
    ('EQ-001', 'LINE-A', 'A', '2026-01-03 22:00:00', 200,  7,  2, 'SCADA'),
    ('EQ-004', 'LINE-C', 'B', '2026-01-04 14:00:00', 188,  9,  8, 'SCADA'),
    ('EQ-005', 'LINE-A', 'C', '2026-01-04 22:00:00', 225,  2,  0, 'SCADA'),
    ('EQ-002', 'LINE-B', 'A', '2026-01-05 06:00:00', 198, 11,  4, 'SCADA'),
    ('EQ-003', 'LINE-C', 'B', '2026-01-05 14:00:00', 212,  6,  3, 'SCADA'),
    -- ⚠️ Anomalias intencionais para demonstrar tratamento de qualidade
    ('EQ-003', 'LINE-A', 'B', '2026-01-03 14:00:00', NULL, 10,  5, 'SCADA'), -- production_qty nulo
    ('EQ-006', 'LINE-B', 'A', '2026-01-03 06:00:00', 215,  -1, 3, 'SCADA'), -- downtime negativo
    (NULL,     'LINE-A', 'C', '2026-01-05 22:00:00', 205,  4,  2, 'SCADA')  -- equipment_id nulo
) AS t(equipment_id, line_id, shift, event_timestamp, production_qty, downtime_minutes, defect_qty, source_system);

## 3. Verificação dos dados ingeridos

Todos os 15 registros devem estar presentes, **incluindo as anomalias**. A Bronze não filtra nada.

In [0]:
%sql
SELECT * FROM bronze.production_events;